# 第12章 選好チューニング

## 12.2 選好チューニングの実装

### 12.2.1 環境の準備

In [ ]:
!pip install datasets transformers[torch,sentencepiece] trl peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.7/105.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 14.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This beha

In [ ]:
from transformers.trainer_utils import set_seed
set_seed(42)

In [ ]:
from google.colab import drive
drive.mount("drive")

### 12.2.2 データセットの準備

In [ ]:
from datasets import load_dataset
dataset_name = "llm-book/alert-preference-2k-ja"
dataset = load_dataset(dataset_name, split="train")

In [ ]:
print(dataset)

In [ ]:
from pprint import pprint
pprint(dataset[0], sort_dicts=False)

In [ ]:
def convert_to_format(example: dict) -> dict:
    """prompr, chosen, rejectedのデータをLLMの入出力として加工"""
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": example["prompt"]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    chosen = example["chosen"] + tokenizer.eos_token
    rejected = example["rejected"] + tokenizer.eos_token
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

In [ ]:
from transformers import AutoTokenizer

base_model_name = "llm-book/Swallow-7b-hf-oasst1-21k-ja"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.padding_side = "left"

In [ ]:
# convert_to_dpo_formatの適応
dataset = dataset.map(
    convert_to_format,
    remove_columns=dataset.colmun_names,
)

In [ ]:
print(dataset)

In [ ]:
print("prompt:\n", dataset[0]["prompt"])
print("chosen:\n", dataset[0]["chosen"])
print("rejected:\n", dataset[0]["rejected"])

In [ ]:
shuffled_dataset = dataset.train_test_split(test_size=0.01)
train_dataset = shuffled_dataset["train"]
eval_dataset = shuffled_dataset["test"]

In [ ]:
print(train_dataset)
print(eval_dataset)